# Load any data (no csv? use this)
| Teacher gives | Use |
|---|---|
| any file (don't know the type) | `load_any("file")` from `clean_tools.py` |
| `.csv` with `;` or tab | `pd.read_csv(f, sep=";")`  /  `sep="\t"` |
| `.txt` with spaces | `pd.read_csv(f, sep=r"\s+")` |
| no header row | `pd.read_csv(f, header=None, names=[...])` |
| `.xlsx` / `.xls` | `pd.read_excel(f, sheet_name=0)` |
| `.json` | `pd.read_json(f)` (or `lines=True`) |
| link (URL) | `pd.read_csv("https://...")` |
| `.zip` | `pd.read_csv("file.zip")` |
| sklearn dataset name (iris, wine, diabetes ...) | `load_xxx(as_frame=True).frame` |
| numbers written in the question | `pd.DataFrame({...})` or paste text into `StringIO` |
| numpy arrays `x`, `y` | `pd.DataFrame(x, columns=[...])` |

Cell 1 makes demo files from titanic so every cell below really runs.

In [1]:
import os, sys, io
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from clean_tools import load_any

os.makedirs("demo_files", exist_ok=True)
t = pd.read_csv("../titanic.csv").head(50)
t.to_csv("demo_files/semicolon.csv", sep=";", index=False)
t.to_csv("demo_files/tab.tsv", sep="\t", index=False)
t[["Age", "Fare", "Pclass"]].dropna().to_csv("demo_files/no_header.txt", sep=" ", index=False, header=False)
t.to_excel("demo_files/data.xlsx", index=False, sheet_name="passengers")
t.to_json("demo_files/data.json", orient="records")
t.to_csv("demo_files/data.zip", index=False, compression="zip")
print(os.listdir("demo_files"))

['data.json', 'data.xlsx', 'data.zip', 'no_header.txt', 'semicolon.csv', 'tab.tsv']


## 1. Don't know the type → `load_any` (guesses everything)

In [2]:
for f in ["semicolon.csv", "tab.tsv", "no_header.txt", "data.xlsx", "data.json", "data.zip"]:
    df = load_any("demo_files/" + f)
    print(f"{f:15s} -> {df.shape}  first cols: {list(df.columns)[:4]}")

semicolon.csv   -> (50, 12)  first cols: ['PassengerId', 'Survived', 'Pclass', 'Name']
tab.tsv         -> (50, 12)  first cols: ['PassengerId', 'Survived', 'Pclass', 'Name']
load_any: no header found -> columns named col0, col1, ...
no_header.txt   -> (36, 3)  first cols: ['col0', 'col1', 'col2']
data.xlsx       -> (50, 12)  first cols: ['PassengerId', 'Survived', 'Pclass', 'Name']
data.json       -> (50, 12)  first cols: ['PassengerId', 'Survived', 'Pclass', 'Name']
data.zip        -> (50, 12)  first cols: ['PassengerId', 'Survived', 'Pclass', 'Name']


## 2. Same files by hand (if you want control)

In [3]:
df = pd.read_csv("demo_files/semicolon.csv", sep=";")                    # ; separated
df = pd.read_csv("demo_files/tab.tsv", sep="\t")                          # tab separated
df = pd.read_csv("demo_files/no_header.txt", sep=r"\s+", header=None,
                 names=["Age", "Fare", "Pclass"])                          # spaces + no header
df = pd.read_excel("demo_files/data.xlsx", sheet_name="passengers")       # Excel (sheet by name or 0)
print(pd.ExcelFile("demo_files/data.xlsx").sheet_names)                   # list the sheets first
df = pd.read_json("demo_files/data.json")                                 # JSON
df = pd.read_csv("demo_files/data.zip")                                   # zip with 1 csv inside
# df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")  # URL
# df = pd.read_csv(r"C:\Users\kowpa\Desktop\exam.csv")                    # full Windows path: r"..."
# Thai text broken?  encoding="utf-8-sig"  or  encoding="cp874"
df.head(3)

['passengers']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


## 3. Teacher says a dataset name → sklearn has it built in (works offline)

In [4]:
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_diabetes, load_digits

data = load_iris(as_frame=True)          # class: 3 flower types
df = data.frame                          # features + "target" column in one table
print(df.shape, "| target names:", [str(n) for n in data.target_names])

print("wine", load_wine(as_frame=True).frame.shape, "(class, 3 types)")
print("breast_cancer", load_breast_cancer(as_frame=True).frame.shape, "(class, 2 types)")
print("diabetes", load_diabetes(as_frame=True).frame.shape, "(regression)")
# needs internet: fetch_california_housing(as_frame=True).frame   (regression, house price)
# any OpenML name:  from sklearn.datasets import fetch_openml; fetch_openml("titanic", version=1, as_frame=True).frame
df.head(3)

(150, 5) | target names: ['setosa', 'versicolor', 'virginica']
wine (178, 14) (class, 3 types)
breast_cancer (569, 31) (class, 2 types)
diabetes (442, 11) (regression)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0


## 4. Numbers written in the question / on the board

In [5]:
# a) type them as a dict
df = pd.DataFrame({
    "hours": [1, 2, 3, 4, 5, 6],
    "sleep": [8, 7, 7, 6, 5, 5],
    "score": [52, 58, 65, 70, 78, 85],
})

# b) paste the table as text (copy from PDF / web) -> StringIO
text = '''
hours,sleep,score
1,8,52
2,7,58
3,7,65
'''
df2 = pd.read_csv(io.StringIO(text))

# c) copied from Excel -> pd.read_clipboard()   (select cells, Ctrl+C, then run)
# df3 = pd.read_clipboard()
print(df2)
df

   hours  sleep  score
0      1      8     52
1      2      7     58
2      3      7     65


,hours,sleep,score
0,1,8,52
1,2,7,58
2,3,7,65
3,4,6,70
4,5,5,78
5,6,5,85


## 5. Teacher gives numpy arrays `X`, `y`

In [6]:
X = np.array([[22, 1], [38, 3], [26, 3], [35, 1]])
y = np.array([7.25, 71.3, 7.9, 53.1])
df = pd.DataFrame(X, columns=["Age", "Pclass"])
df["Fare"] = y
# Q1 (from scratch) can also use X, y directly: x_train, x_test = X[:n], X[n:]
df

,Age,Pclass,Fare
0,22,1,7.25
1,38,3,71.30
2,26,3,7.90
3,35,1,53.10


## 6. Two files: train + test

In [7]:
train = load_any("demo_files/semicolon.csv")
test = load_any("demo_files/data.json")
both = pd.concat([train, test], keys=["train", "test"])   # clean both the same way, then split back
train2, test2 = both.loc["train"], both.loc["test"]
print(train2.shape, test2.shape)

(50, 12) (50, 12)
